In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher

In [ ]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from numba import njit, prange
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import os
import json
import sys
import gc

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from sklearn.preprocessing import StandardScaler

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [ ]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [ ]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [ ]:
index_mapping = {
    # Indices
    "Nifty": {
        "symbol": "NSE:NIFTY50-INDEX",
        "quantity": 75
    },
    "Bank_Nifty": {
        "symbol": "NSE:NIFTYBANK-INDEX",
        "quantity": 30
    },
    "Finnifty": {
        "symbol": "NSE:FINNIFTY-INDEX",
        "quantity": 40
    },
    "Sensex": {
        "symbol": "BSE:SENSEX-INDEX",
        "quantity": 20
    },
    "Bankex": {
        "symbol": "BSE:BANKEX-INDEX",
        "quantity": 15
    },

    # Complete Nifty 50 Stocks

    "Adani_Enterprises_Ltd": {
        "symbol": "NSE:ADANIENT-EQ",
        "quantity": 10
    },
    "Adani_Ports__SEZ_Ltd": {
        "symbol": "NSE:ADANIPORTS-EQ",
        "quantity": 10
    },
    "Apollo_Hospitals_Enterprise_Ltd": {
        "symbol": "NSE:APOLLOHOSP-EQ",
        "quantity": 10
    },
    "Asian_Paints_Ltd": {
        "symbol": "NSE:ASIANPAINT-EQ",
        "quantity": 10
    },
    "Axis_Bank_Ltd": {
        "symbol": "NSE:AXISBANK-EQ",
        "quantity": 10
    },
    "Bajaj_Auto_Ltd": {
        "symbol": "NSE:BAJAJ-AUTO-EQ",
        "quantity": 10
    },
    "Bajaj_Finance_Ltd": {
        "symbol": "NSE:BAJFINANCE-EQ",
        "quantity": 10
    },
    "Bajaj_Finserv_Ltd": {
        "symbol": "NSE:BAJAJFINSV-EQ",
        "quantity": 10
    },
    "Bharat_Electronics_Ltd": {
        "symbol": "NSE:BEL-EQ",
        "quantity": 10
    },
    "Bharti_Airtel_Ltd": {
        "symbol": "NSE:BHARTIARTL-EQ",
        "quantity": 10
    },
    "BPCL_Ltd": {
        "symbol": "NSE:BPCL-EQ",
        "quantity": 10
    },
    "Cipla_Ltd": {
        "symbol": "NSE:CIPLA-EQ",
        "quantity": 10
    },
    "Coal_India_Ltd": {
        "symbol": "NSE:COALINDIA-EQ",
        "quantity": 10
    },
    "Dr_Reddys_Laboratories_Ltd": {
        "symbol": "NSE:DRREDDY-EQ",
        "quantity": 10
    },
    "Eicher_Motors_Ltd": {
        "symbol": "NSE:EICHERMOT-EQ",
        "quantity": 10
    },
    "Grasim_Industries_Ltd": {
        "symbol": "NSE:GRASIM-EQ",
        "quantity": 10
    },
    "HCL_Technologies_Ltd": {
        "symbol": "NSE:HCLTECH-EQ",
        "quantity": 10
    },
    "HDFC_Bank_Ltd": {
        "symbol": "NSE:HDFCBANK-EQ",
        "quantity": 10
    },
    "HDFC_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:HDFCLIFE-EQ",
        "quantity": 10
    },
    "Hero_MotoCorp_Ltd": {
        "symbol": "NSE:HEROMOTOCO-EQ",
        "quantity": 10
    },
    "Hindalco_Industries_Ltd": {
        "symbol": "NSE:HINDALCO-EQ",
        "quantity": 10
    },
    "Hindustan_Unilever_Ltd": {
        "symbol": "NSE:HINDUNILVR-EQ",
        "quantity": 10
    },
    "ICICI_Bank_Ltd": {
        "symbol": "NSE:ICICIBANK-EQ",
        "quantity": 10
    },
    "IndusInd_Bank_Ltd": {
        "symbol": "NSE:INDUSINDBK-EQ",
        "quantity": 10
    },
    "Infosys_Ltd": {
        "symbol": "NSE:INFY-EQ",
        "quantity": 10
    },
    "Indian_Oil_Corporation_Ltd": {
        "symbol": "NSE:IOC-EQ",
        "quantity": 10
    },
    "JSW_Steel_Ltd": {
        "symbol": "NSE:JSWSTEEL-EQ",
        "quantity": 10
    },
    "Kotak_Mahindra_Bank_Ltd": {
        "symbol": "NSE:KOTAKBANK-EQ",
        "quantity": 10
    },
    "Larsen__Toubro_Ltd": {
        "symbol": "NSE:LT-EQ",
        "quantity": 10
    },
    "Mahindra__Mahindra_Ltd": {
        "symbol": "NSE:M&M-EQ",
        "quantity": 10
    },
    "Maruti_Suzuki_India_Ltd": {
        "symbol": "NSE:MARUTI-EQ",
        "quantity": 10
    },
    "Nestle_India_Ltd": {
        "symbol": "NSE:NESTLEIND-EQ",
        "quantity": 10
    },
    "NTPC_Ltd": {
        "symbol": "NSE:NTPC-EQ",
        "quantity": 10
    },
    "ONGC_Ltd": {
        "symbol": "NSE:ONGC-EQ",
        "quantity": 10
    },
    "Power_Grid_Corporation_of_India_Ltd": {
        "symbol": "NSE:POWERGRID-EQ",
        "quantity": 10
    },
    "Reliance_Industries_Ltd": {
        "symbol": "NSE:RELIANCE-EQ",
        "quantity": 10
    },
    "State_Bank_of_India_(SBI)_Ltd": {
        "symbol": "NSE:SBIN-EQ",
        "quantity": 10
    },
    "SBI_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:SBILIFE-EQ",
        "quantity": 10
    },
    "Sun_Pharmaceutical_Industries_Ltd": {
        "symbol": "NSE:SUNPHARMA-EQ",
        "quantity": 10
    },
    "Tata_Consumer_Products_Ltd": {
        "symbol": "NSE:TATACONSUM-EQ",
        "quantity": 10
    },
    "Tata_Motors_Ltd": {
        "symbol": "NSE:TATAMOTORS-EQ",
        "quantity": 10
    },
    "Tata_Steel_Ltd": {
        "symbol": "NSE:TATASTEEL-EQ",
        "quantity": 10
    },
    "TCS_Ltd": {
        "symbol": "NSE:TCS-EQ",
        "quantity": 10
    },
    "Tech_Mahindra_Ltd": {
        "symbol": "NSE:TECHM-EQ",
        "quantity": 10
    },
    "Titan_Company_Ltd": {
        "symbol": "NSE:TITAN-EQ",
        "quantity": 10
    },
    "Trent_Ltd": {
        "symbol": "NSE:TRENT-EQ",
        "quantity": 10
    },
    "UltraTech_Cement_Ltd": {
        "symbol": "NSE:ULTRACEMCO-EQ",
        "quantity": 10
    },
    "Wipro_Ltd": {
        "symbol": "NSE:WIPRO-EQ",
        "quantity": 10
    }
}

In [ ]:
# Historical config for fetching and saving raw data
history_config = {
    "timeframes": [1, 2, 3, 5, 10, 15, 20, 30, 45, 60, 120, 180, 240],
    "colab_folder_path": "/content/drive/MyDrive/historical_data",
    "colab_folder_path_processed": "processed_data",
    "local_folder_path": "historical_data",
    "local_folder_path_processed": "C:\\Users\\iamma\\OneDrive\\Desktop\\Marshal\\processed_data",
    "bar_limit": 15  # parameter for fetch_train_candle_data
}

In [ ]:
if "google.colab" in sys.modules:
    folder_path = history_config["colab_folder_path"]
else:
    folder_path = history_config["local_folder_path"]

In [ ]:
# Ensure the folder exists
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

In [ ]:
# Main loop to fetch and save data
for instrument_name, info in index_mapping.items():
    for tf in history_config["timeframes"]:
        file_name = f"{instrument_name}_{tf}.csv"
        file_path = os.path.join(folder_path, file_name)

        # Skip if file already exists
        if os.path.exists(file_path):
            print(f"File already exists for {instrument_name} timeframe {tf}. Skipping.")
            continue

        # Fetch data for given bar_limit, symbol, and timeframe
        df = fetch_train_candle_data(history_config["bar_limit"], info["symbol"], tf)

        # Save if data is not empty
        if not df.empty:
            df.to_csv(file_path, index=False)
            print(f"Saved data for {instrument_name} timeframe {tf} at {file_path}")
        else:
            print(f"No data returned for {instrument_name} timeframe {tf}.")

File already exists for Nifty timeframe 1. Skipping.
File already exists for Nifty timeframe 2. Skipping.
File already exists for Nifty timeframe 3. Skipping.
File already exists for Nifty timeframe 5. Skipping.
File already exists for Nifty timeframe 10. Skipping.
File already exists for Nifty timeframe 15. Skipping.
File already exists for Nifty timeframe 20. Skipping.
File already exists for Nifty timeframe 30. Skipping.
File already exists for Nifty timeframe 45. Skipping.
File already exists for Nifty timeframe 60. Skipping.
File already exists for Nifty timeframe 120. Skipping.
File already exists for Nifty timeframe 180. Skipping.
File already exists for Nifty timeframe 240. Skipping.
File already exists for Bank_Nifty timeframe 1. Skipping.
File already exists for Bank_Nifty timeframe 2. Skipping.
File already exists for Bank_Nifty timeframe 3. Skipping.
File already exists for Bank_Nifty timeframe 5. Skipping.
File already exists for Bank_Nifty timeframe 10. Skipping.
File alr

In [ ]:
@njit
def label_signals_jit(close_arr, high_arr, low_arr, target_arr, stoploss_arr):
    n = len(close_arr)
    signals = np.zeros(n, dtype=np.float32)
    entry_prices = np.zeros(n, dtype=np.float32)
    exit_prices = np.zeros(n, dtype=np.float32)
    candles_to_profit = np.zeros(n, dtype=np.float32)
    candles_to_loss = np.zeros(n, dtype=np.float32)

    for i in range(n - 1):
        entry_price = close_arr[i]
        target = target_arr[i]
        stop_loss = stoploss_arr[i]

        buy_target_price = entry_price + target
        buy_sl_price = entry_price - stop_loss
        sell_target_price = entry_price - target
        sell_sl_price = entry_price + stop_loss

        signal_found = False
        for offset in range(i + 1, n):
            future_high = high_arr[offset]
            future_low = low_arr[offset]

            triggers = []
            # Check Buy triggers
            if future_high >= buy_target_price:
                triggers.append((1, offset - i))  # buy target
            if future_low <= buy_sl_price:
                triggers.append((2, offset - i))  # buy stoploss
            # Check Sell triggers
            if future_low <= sell_target_price:
                triggers.append((3, offset - i))  # sell target
            if future_high >= sell_sl_price:
                triggers.append((4, offset - i))  # sell stoploss

            # If any triggers fired, pick the first based on priority (1,2,3,4)
            if triggers:
                triggers.sort(key=lambda x: x[0])
                first_trigger, candle_count = triggers[0]

                signals[i] = float(first_trigger)
                entry_prices[i] = entry_price

                if first_trigger == 1:  # buy target
                    exit_prices[i] = buy_target_price
                    candles_to_profit[i] = float(candle_count)
                elif first_trigger == 2:  # buy stoploss
                    exit_prices[i] = buy_sl_price
                    candles_to_loss[i] = float(candle_count)
                elif first_trigger == 3:  # sell target
                    exit_prices[i] = sell_target_price
                    candles_to_profit[i] = float(candle_count)
                elif first_trigger == 4:  # sell stoploss
                    exit_prices[i] = sell_sl_price
                    candles_to_loss[i] = float(candle_count)

                signal_found = True
                break

        if not signal_found:
            signals[i] = 0
            entry_prices[i] = entry_price

    return signals, entry_prices, exit_prices, candles_to_profit, candles_to_loss


class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns: [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')
        self.df['datetime'] = (
            pd.to_datetime(self.df['datetime'], unit='s')
            .dt.tz_localize('UTC')
            .dt.tz_convert(ist)
            .dt.tz_localize(None)
        )

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        if 'volume' in self.df.columns:
            vol_condition = self.df['volume'].isnull() | (self.df['volume'] <= 0)
            if vol_condition.any():
                self.df.drop('volume', axis=1, inplace=True)
        return self

    def add_indicator_features(self):
        # ATR
        self.df['atr_14'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)

        # RSI
        self.df['rsi_14'] = ta.rsi(
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)

        # Stoch
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14,
            d=3
        ).astype(np.float32).round(2)
        self.df = self.df.join(stoch)

        # MACD
        macd = ta.macd(
            close=self.df['close'],
            fast=12,
            slow=26,
            signal=9
        ).astype(np.float32).round(2)
        self.df = self.df.join(macd)

        # ADX
        adx = ta.adx(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)
        self.df = self.df.join(adx[['ADX_14']])
        self.df.rename(columns={'ADX_14': 'adx_14'}, inplace=True)

        # EMAs
        self.df['ema_50'] = ta.ema(self.df['close'], length=50).astype(np.float32).round(2)
        self.df['ema_200'] = ta.ema(self.df['close'], length=200).astype(np.float32).round(2)

        # SMA
        self.df['sma_20'] = ta.sma(self.df['close'], length=20).astype(np.float32).round(2)

        # Keltner Channels
        kc = ta.kc(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=20
        ).astype(np.float32).round(2)
        self.df = self.df.join(kc)

        # Volume-based
        if 'volume' in self.df.columns:
            self.df['obv'] = ta.obv(self.df['close'], self.df['volume']).astype(np.float32).round(2)


        # Price Action Features
        self.df['price_range'] = (self.df['high'] - self.df['low']).astype(np.float32).round(2)
        self.df['body_size'] = (self.df['close'] - self.df['open']).abs().astype(np.float32).round(2)
        self.df['upper_wick'] = (
            self.df['high'] - self.df[['open', 'close']].max(axis=1)
        ).astype(np.float32).round(2)
        self.df['lower_wick'] = (
            self.df[['open', 'close']].min(axis=1) - self.df['low']
        ).astype(np.float32).round(2)

        return self

    def add_time_features(self):
        # Hour, day_of_week, month
        self.df['hour'] = self.df.index.hour.astype(np.int32)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(np.int32)
        self.df['month'] = self.df.index.month.astype(np.int32)
        return self

    def add_adaptive_targets_and_stops(self):
        # ATR-based target and stoploss
        self.df['Target'] = (2.0 * self.df['atr_14']).astype(np.float32).round(2)
        self.df['StopLoss'] = (1.0 * self.df['atr_14']).astype(np.float32).round(2)
        return self

    def label_signals(self):
        from __main__ import label_signals_jit  # or from the same notebook cell, as needed

        close_arr = self.df['close'].values
        high_arr = self.df['high'].values
        low_arr = self.df['low'].values
        target_arr = self.df['Target'].values
        stoploss_arr = self.df['StopLoss'].values

        signals, entry_prices, exit_prices, ctp, ctl = label_signals_jit(
            close_arr,
            high_arr,
            low_arr,
            target_arr,
            stoploss_arr
        )
        self.df['Signal'] = signals.astype(np.float32)
        self.df['Entry Price'] = entry_prices.astype(np.float32).round(2)
        self.df['Exit Price'] = exit_prices.astype(np.float32).round(2)
        self.df['candles_to_profit'] = ctp.astype(np.float32)
        self.df['candles_to_loss'] = ctl.astype(np.float32)

        return self

    def run_pipeline(self):
        (self.preprocess_datetime()
             .clean_data()
             .add_indicator_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
             .label_signals())
        return self

    def get_processed_df(self):
        # Drop rows with any NaN
        self.df.dropna(axis=0, how='any', inplace=True)

        # Remove Entry/Exit Price columns if you don't need them
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        return self.df

In [ ]:
if "google.colab" in sys.modules:
    processed_folder_path = history_config["colab_folder_path_processed"]
else:
    processed_folder_path = history_config["local_folder_path_processed"]

if not os.path.exists(processed_folder_path):
    os.makedirs(processed_folder_path)

In [ ]:
# We'll store (instrument_name, timeframe, processed_df) tuples in this list
temp_data_list = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)

        # Split the filename on the last underscore to accommodate multiple underscores in the instrument name
        instrument_name, timeframe_str = file.replace(".csv", "").rsplit("_", 1)
        timeframe = int(timeframe_str)

        processed_file_name = f"{instrument_name}_{timeframe}_processed.csv"
        processed_file_path = os.path.join(processed_folder_path, processed_file_name)

        # If processed file exists, skip reprocessing
        if os.path.exists(processed_file_path):
            print(f"Processed file already exists for {instrument_name} {timeframe}M. Skipping.")
        else:
            # Process it once, then drop from memory
            raw_df = pd.read_csv(file_path).drop_duplicates(subset='datetime', keep='first')
            pipeline = FullFeaturePipeline(raw_df)
            pipeline.run_pipeline()
            processed_df = pipeline.get_processed_df()

            processed_df.to_csv(processed_file_path, index=True, index_label='datetime')
            print(f"Processed and saved data for {instrument_name} {timeframe}M at {processed_file_path}")

            # Cleanup to free memory
            del raw_df
            del processed_df
            gc.collect()

        # We always add the path so we know it exists (either newly processed or previously)
        temp_data_list.append((instrument_name, timeframe, processed_file_path))

Processed file already exists for Adani_Ports__SEZ_Ltd 240M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 60M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 10M. Skipping.
Processed file already exists for Adani_Ports__SEZ_Ltd 180M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 240M. Skipping.
Processed file already exists for Adani_Ports__SEZ_Ltd 20M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 45M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 5M. Skipping.
Processed file already exists for Adani_Ports__SEZ_Ltd 15M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 30M. Skipping.
Processed file already exists for Adani_Ports__SEZ_Ltd 120M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 180M. Skipping.
Processed file already exists for Adani_Enterprises_Ltd 3M. Skipping.
Processed file already exists for Adani_Ports__SEZ_Ltd 1M. Skipping.
Processed 

In [ ]:
# Sort by timeframe descending, then by instrument name alphabetically
temp_data_list.sort(key=lambda x: (-x[1], x[0]))

instruments_config = []
for i, (instrument_name, timeframe, csv_path) in enumerate(temp_data_list):
    lot_size = index_mapping[instrument_name]["quantity"]
    dynamic_name = f"{instrument_name.upper()}_{timeframe}M"

    instruments_config.append({
        # Here we explicitly assign each instrument a unique ID
        "instrument_id": i,
        "name": dynamic_name,
        "file_path": csv_path,
        "lot_size": lot_size,
        "transaction_cost": 20.0  # default brokerage
    })

In [ ]:
# Now build the main config dictionary
config = {
    "instruments": instruments_config,
    "window_size": 30,
    "unrealized_pnl_weight": 0.1,
    "time_penalty_weight": 0.001,
    "volatility_penalty_weight": 0.05,
    "target_bonus": 0.5,
    "sl_penalty": 0.5,
    "misalignment_penalty": 0.5,
    "missed_opportunity_penalty": 0.05,
    "target_proximity_weight": 0.2,
    "sl_proximity_weight": 0.05,
    "max_step_reward": 5,

    # Trailing Stop Parameters
    "enable_trailing_sl": True,
    "trailing_sl_mode": "factor",   # can be "fixed", "percentage", or "factor"
    "trailing_sl_amount": 20.0,
    "trailing_sl_percent": 0.01,
    "trailing_factor": 0.5,

    "initial_cap_multiplier": 10,   # factor for initial capital
    "normalize_data": True          # whether to normalize data for the agent's observation
}

# Margin constants
MARGIN_FRACTION = 0.05
MARGIN_CALL_THRESHOLD = 0.5

In [ ]:
# Print a summary of the dynamically created instruments
print("Final Config Instruments:")
for inst in config["instruments"]:
    print(f"{inst['instrument_id']}: {inst['name']}, lot_size={inst['lot_size']}, brokerage={inst['transaction_cost']}")

Final Config Instruments:
0: ADANI_ENTERPRISES_LTD_240M, lot_size=10, brokerage=20.0
1: ADANI_PORTS__SEZ_LTD_240M, lot_size=10, brokerage=20.0
2: APOLLO_HOSPITALS_ENTERPRISE_LTD_240M, lot_size=10, brokerage=20.0
3: ASIAN_PAINTS_LTD_240M, lot_size=10, brokerage=20.0
4: AXIS_BANK_LTD_240M, lot_size=10, brokerage=20.0
5: BPCL_LTD_240M, lot_size=10, brokerage=20.0
6: BAJAJ_AUTO_LTD_240M, lot_size=10, brokerage=20.0
7: BAJAJ_FINANCE_LTD_240M, lot_size=10, brokerage=20.0
8: BAJAJ_FINSERV_LTD_240M, lot_size=10, brokerage=20.0
9: BANK_NIFTY_240M, lot_size=30, brokerage=20.0
10: BANKEX_240M, lot_size=15, brokerage=20.0
11: BHARAT_ELECTRONICS_LTD_240M, lot_size=10, brokerage=20.0
12: BHARTI_AIRTEL_LTD_240M, lot_size=10, brokerage=20.0
13: CIPLA_LTD_240M, lot_size=10, brokerage=20.0
14: COAL_INDIA_LTD_240M, lot_size=10, brokerage=20.0
15: DR_REDDYS_LABORATORIES_LTD_240M, lot_size=10, brokerage=20.0
16: EICHER_MOTORS_LTD_240M, lot_size=10, brokerage=20.0
17: FINNIFTY_240M, lot_size=40, brokerage=2

In [ ]:
def get_dataframe_by_name(instrument_name, config):
    # instrument_name is something like "BANKNIFTY_5M", "NIFTY50_15M", etc.
    for instrument in config["instruments"]:
        if instrument["name"] == instrument_name:
            return pd.read_csv(instrument["file_path"], parse_dates=['datetime'], index_col='datetime')
    return None


my_df = get_dataframe_by_name("BANK_NIFTY_5M", config)

my_df

,open,high,low,close,atr_14,rsi_14,STOCHk_14_3_3,STOCHd_14_3_3,MACD_12_26_9,MACDh_12_26_9,...,upper_wick,lower_wick,hour,day_of_week,month,Target,StopLoss,Signal,candles_to_profit,candles_to_loss
datetime,,,,,,,,,,,,,,,,,,,,,
2020-09-30 13:20:00,21360.20,21360.70,21340.90,21353.30,49.23,53.68,64.36,69.07,16.04,4.00,...,0.50,12.40,13,2,9,98.46,49.23,2.0,0.0,1.0
2020-09-30 13:25:00,21352.70,21378.80,21284.10,21295.80,52.48,45.53,56.08,62.38,10.83,-0.97,...,26.10,11.70,13,2,9,104.96,52.48,4.0,0.0,1.0
2020-09-30 13:30:00,21294.40,21354.20,21277.80,21346.60,54.19,52.40,51.33,57.25,10.67,-0.90,...,7.60,16.60,13,2,9,108.38,54.19,4.0,0.0,4.0
2020-09-30 13:35:00,21347.90,21351.50,21309.10,21319.10,53.35,48.81,39.66,49.02,8.23,-2.67,...,3.60,10.00,13,2,9,106.70,53.35,4.0,0.0,2.0
2020-09-30 13:40:00,21322.80,21334.50,21294.70,21329.50,52.38,50.20,41.35,44.11,7.06,-3.08,...,5.00,28.10,13,2,9,104.76,52.38,4.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-06 15:05:00,52311.15,52315.55,52289.20,52300.80,40.16,44.48,37.44,30.19,-15.79,-5.99,...,4.40,11.60,15,2,11,80.32,40.16,4.0,0.0,2.0
2024-11-06 15:10:00,52303.55,52317.15,52282.90,52294.55,39.74,43.34,39.41,36.14,-15.93,-4.90,...,13.60,11.65,15,2,11,79.48,39.74,4.0,0.0,1.0
2024-11-06 15:15:00,52294.05,52357.65,52293.75,52326.40,41.47,50.32,42.30,39.72,-13.31,-1.83,...,31.25,0.30,15,2,11,82.94,41.47,4.0,0.0,2.0


In [ ]:
class Position:
    # Holds all necessary info about the current position (long or short).
    def __init__(self):
        self.reset()

    def reset(self):
        self.status = "flat"             # "flat", "long", or "short"
        self.entry_price = None
        self.sl_points = None            # For initial SL
        self.target_points = None
        self.direction = 0               # +1 for long, -1 for short
        self.quantity = 0
        self.time_in_trade = 0
        self.margin_used = 0.0           # Tracks margin used for the current position

        # Trailing SL related
        self.trailing_sl_price = None    # Absolute trailing SL price
        self.highest_price = None        # For tracking highest price since entry (long)
        self.lowest_price = None         # For tracking lowest price since entry (short)

    def update_unrealized_pnl(self, current_price: float) -> float:
        if self.status == "flat":
            return 0.0
        return (current_price - self.entry_price) * self.direction * self.quantity

    def open(self, entry_price: float, sl_points: float, target_points: float,
             direction: int, quantity: int):
        self.status = "long" if direction == 1 else "short"
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.direction = direction
        self.quantity = quantity
        self.time_in_trade = 0

        # Initialize trailing stop to the initial SL directly
        if direction == 1:
            self.trailing_sl_price = entry_price - sl_points
            self.highest_price = entry_price
            self.lowest_price = None
        else:
            self.trailing_sl_price = entry_price + sl_points
            self.highest_price = None
            self.lowest_price = entry_price

In [ ]:

class TradingEnv(gym.Env):
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        self.config = env_config
        self.total_reward = 0.0

        # Mandatory parameters
        self.window_size = self.config["window_size"]
        self.lot_size = self.config["lot_size"]
        self.transaction_cost = self.config["transaction_cost"]
        self.instrument_name = self.config.get("name", "Unknown")
        self.instrument_id = self.config.get("instrument_id", 0)
        self.normalize_data = self.config.get("normalize_data", False)

        # Chunk-related parameters
        self.csv_path = self.config["file_path"]
        self.chunksize = self.config.get("chunksize", 50000)
        self.chunk_overlap = self.window_size

        # Prepare a partial-fit scaler for normalization
        if self.normalize_data:
            self.scaler = StandardScaler()
            self._partial_fit_scaler()
        else:
            self.scaler = None

        # Buffers for data
        self.buffer_raw = pd.DataFrame()
        self.buffer_guidance = pd.DataFrame()
        self.buffer_obs = pd.DataFrame()

        # Chunking trackers
        self.current_chunk_start = 0
        self.current_chunk_end = 0
        self.total_rows = self._count_csv_rows(self.csv_path)

        # Precompute max high for initial capital
        self.max_high = self._compute_max_high()
        multiplier = self.config.get("initial_cap_multiplier", 3)
        self.initial_capital = self.max_high * self.lot_size * MARGIN_FRACTION * multiplier
        self.current_capital = None
        self.current_step = None

        self.position = Position()
        self.trade_log = []

        # Action space
        self.action_space = spaces.Discrete(4)

        # We finalize the Box shapes after loading the first chunk
        self.observation_space = None

        # Reward shaping parameters
        self.unrealized_pnl_weight = self.config.get("unrealized_pnl_weight", 0.1)
        self.time_penalty_weight = self.config.get("time_penalty_weight", 0.02)
        self.volatility_penalty_weight = self.config.get("volatility_penalty_weight", 0.1)
        self.target_bonus = self.config.get("target_bonus", 0.5)
        self.sl_penalty = self.config.get("sl_penalty", 0.5)
        self.misalignment_penalty = self.config.get("misalignment_penalty", 0.5)
        self.missed_opportunity_penalty = self.config.get("missed_opportunity_penalty", 0.05)
        self.target_proximity_weight = self.config.get("target_proximity_weight", 0.2)
        self.sl_proximity_weight = self.config.get("sl_proximity_weight", 0.05)
        self.max_step_reward = self.config.get("max_step_reward", 5.0)

        # Trailing SL parameters
        self.enable_trailing_sl = self.config.get("enable_trailing_sl", True)
        self.trailing_sl_mode = self.config.get("trailing_sl_mode", "fixed")
        self.trailing_sl_amount = self.config.get("trailing_sl_amount", 20.0)
        self.trailing_sl_percent = self.config.get("trailing_sl_percent", 0.01)
        self.trailing_factor = self.config.get("trailing_factor", 0.5)

        # Load the first chunk to define the observation space
        # Force loading around index=0
        self._load_next_chunk_if_needed(target_index=0, force=True)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = self.window_size
        if self.current_step >= self.total_rows:
            # Edge case: CSV smaller than window_size
            self.current_step = max(0, self.total_rows - 1)

        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log.clear()
        self.total_reward = 0.0

        # Force load chunk that at least covers [self.current_step - window_size, self.current_step]
        self._load_next_chunk_if_needed(target_index=self.current_step, force=True)

        obs = self._get_observation()
        return obs, {}

    def step(self, action: int):
        # Ensure correct chunk is loaded for current_step
        self._load_next_chunk_if_needed(self.current_step)

        current_data = self._get_row(self.buffer_raw, self.current_step)
        guidance_row = self._get_row(self.buffer_guidance, self.current_step)
        reward = 0.0

        # Guidance reward if flat
        if self.position.status == "flat":
            reward += self._apply_guidance_reward(action, guidance_row)

        # Check margin call
        if self.position.status != "flat":
            if self._check_margin_call(current_data['close']):
                reward += self._close_position(current_data, "Margin Call")

        # Update trailing SL
        if self.enable_trailing_sl and self.position.status != "flat":
            self._update_trailing_sl(current_data)

        # SL hit check
        hit_sl, fill_price = self._check_sl_hit(current_data)
        if hit_sl:
            reward += self._close_position(current_data, "SL/Trailing SL Hit", override_exit_price=fill_price)

        # Execute action
        self._execute_action(action, current_data)

        # Reward shaping if in an open position
        if self.position.status != "flat":
            self.position.time_in_trade += 1
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
            reward += self._calculate_reward(unrealized_pnl, current_data)

        # Advance step
        self.current_step += 1
        if self.current_step >= self.total_rows or self.current_capital <= 0:
            done = True
            # Force close if still open
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")
        else:
            done = False

        self.total_reward += reward
        obs = self._get_observation()
        return obs, reward, done, False, {}

    # =============================
    # CHUNKING & LOADING
    # =============================
    def _partial_fit_scaler(self):
        # Partial-fit the scaler on the entire CSV
        reader = pd.read_csv(self.csv_path, parse_dates=['datetime'], index_col='datetime', chunksize=self.chunksize)
        for chunk in reader:
            chunk_obs = chunk.drop(columns=['Signal','candles_to_profit','candles_to_loss','StopLoss','Target'],
                                   errors='ignore')
            chunk_obs = chunk_obs.ffill().bfill()
            self.scaler.partial_fit(chunk_obs.values)

    def _count_csv_rows(self, csv_path):
        # Quick way to count lines minus header
        with open(csv_path, 'r') as f:
            return sum(1 for _ in f) - 1

    def _compute_max_high(self):
        max_val = float('-inf')
        reader = pd.read_csv(self.csv_path, parse_dates=['datetime'], index_col='datetime', chunksize=self.chunksize)
        for chunk in reader:
            if 'high' in chunk.columns:
                local_max = chunk['high'].max()
                if local_max > max_val:
                    max_val = local_max
        if max_val == float('-inf'):
            max_val = 1.0  # fallback if CSV has no 'high' column
        return max_val

    def _load_next_chunk_if_needed(self, target_index, force=False):
        """
        Ensures self.buffer_* have enough rows to cover [target_index - window_size, target_index].
        If 'force' is True, we always reload from scratch.
        If 'force' is False, we only reload if target_index is outside the current buffer range.
        """
        # If already in range and not forced, do nothing
        if not force:
            if (target_index >= self.current_chunk_start) and (target_index < self.current_chunk_end):
                # Also check that (target_index - window_size) is in the chunk
                needed_start = target_index - self.window_size
                if needed_start < 0:
                    needed_start = 0
                if needed_start >= self.current_chunk_start:
                    return

        # Clear the buffers
        self.buffer_raw = pd.DataFrame()
        self.buffer_guidance = pd.DataFrame()
        self.buffer_obs = pd.DataFrame()

        self.current_chunk_start = 0
        self.current_chunk_end = 0

        # We want to ensure [target_index - window_size, target_index] is loaded
        needed_start = max(0, target_index - self.window_size)

        # Read from CSV chunk by chunk, until we pass target_index row
        start_reader = pd.read_csv(self.csv_path, parse_dates=['datetime'], index_col='datetime', chunksize=self.chunksize)

        running_count = 0
        for chunk in start_reader:
            chunk_len = len(chunk)
            chunk_start = running_count
            chunk_end = running_count + chunk_len

            # Basic columns
            chunk_guidance = chunk[['Signal','candles_to_profit','candles_to_loss']].copy() if 'Signal' in chunk.columns else pd.DataFrame()
            chunk_raw = chunk.drop(columns=['Signal','candles_to_profit','candles_to_loss'], errors='ignore').copy()

            # Exclude columns from obs
            exclude_from_obs = ["StopLoss","Target"]
            chunk_obs = chunk_raw.drop(columns=exclude_from_obs, errors='ignore').copy()

            # Fill missing
            chunk_raw = chunk_raw.ffill().bfill()
            chunk_guidance = chunk_guidance.ffill().bfill()
            chunk_obs = chunk_obs.ffill().bfill()

            # Scale if needed
            if self.normalize_data and self.scaler is not None:
                arr = chunk_obs.values
                transformed = self.scaler.transform(arr)
                chunk_obs = pd.DataFrame(transformed, columns=chunk_obs.columns, index=chunk_obs.index)

            # Concat
            self.buffer_raw = pd.concat([self.buffer_raw, chunk_raw])
            self.buffer_guidance = pd.concat([self.buffer_guidance, chunk_guidance])
            self.buffer_obs = pd.concat([self.buffer_obs, chunk_obs])

            self.current_chunk_end = chunk_end
            running_count += chunk_len

            # Once we have read enough rows to definitely cover target_index, break
            if target_index < running_count:
                break

        # Now we slice off from needed_start to whatever we have
        # This ensures we keep an overlap for the entire window
        if needed_start < self.current_chunk_end:
            self.current_chunk_start = needed_start
        else:
            self.current_chunk_start = max(0, self.current_chunk_end - 1)

        # Slice data frames to keep only [current_chunk_start : ]
        self.buffer_raw = self.buffer_raw.iloc[self.current_chunk_start:]
        self.buffer_guidance = self.buffer_guidance.iloc[self.current_chunk_start:]
        self.buffer_obs = self.buffer_obs.iloc[self.current_chunk_start:]

        # Recompute new start as an absolute index
        new_length = len(self.buffer_raw)
        self.current_chunk_start = self.current_chunk_end - new_length

        # If observation_space still None, define it
        if self.observation_space is None:
            n_features = self.buffer_obs.shape[1]
            self.observation_space = spaces.Dict({
                "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
                "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
                "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32),
                "instrument_info": spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
            })

    def _get_row(self, df: pd.DataFrame, index: int) -> pd.Series:
        """
        Safely fetch a row from df, returning a zero-filled Series if out-of-bounds.
        """
        local_idx = index - self.current_chunk_start
        if local_idx < 0 or local_idx >= len(df):
            # Return zeros if out of chunk bounds to avoid IndexError
            if df.shape[1] == 0:
                # If df is empty, just return a dummy
                return pd.Series([], dtype=np.float32)
            return pd.Series(np.zeros(df.shape[1], dtype=np.float32), index=df.columns)
        return df.iloc[local_idx]

    # ==============
    # OBSERVATIONS
    # ==============
    def _get_observation(self) -> dict:
        # Make sure chunk covers current_step
        self._load_next_chunk_if_needed(self.current_step)

        idx = self.current_step
        row_obs = self._get_row(self.buffer_obs, idx).values.astype(np.float32)

        current_close = self._get_row(self.buffer_raw, idx)['close']
        unrealized_pnl = self.position.update_unrealized_pnl(current_close)
        normalized_pnl = unrealized_pnl / self.initial_capital
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            normalized_pnl,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        # Build rolling history
        # For i in [idx - window_size, ..., idx-1]
        history_data = []
        start_idx = idx - self.window_size
        for i in range(start_idx, idx):
            if i < 0 or i < self.current_chunk_start or i >= self.current_chunk_end:
                # Append zero row
                history_data.append(np.zeros_like(row_obs, dtype=np.float32))
            else:
                h_row = self._get_row(self.buffer_obs, i).values.astype(np.float32)
                history_data.append(h_row)
        history_data = np.array(history_data, dtype=np.float32)

        normalized_lot_size = self.lot_size / 1000.0
        normalized_transaction_cost = self.transaction_cost / 100.0
        instrument_info = np.array([normalized_lot_size, normalized_transaction_cost], dtype=np.float32)

        return {
            "market": row_obs,
            "position": position_context,
            "history": history_data,
            "instrument_info": instrument_info
        }

    # ==============
    # ACTIONS
    # ==============
    def _execute_action(self, action: int, current_data: pd.Series):
        price = current_data['open'] if 'open' in current_data else 0.0
        # action = 0 => hold, 1 => long, 2 => short, 3 => manual close

        if action == 3 and self.position.status != "flat":
            self._close_position(current_data, "Manual Close")

        elif action == 1:
            # if currently short, close it first
            if self.position.status == "short":
                self._close_position(current_data, "Reverse Close")

            margin_cost = price * self.lot_size * MARGIN_FRACTION
            if self.position.status == "flat" and self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("long", current_data)

        elif action == 2:
            # if currently long, close it first
            if self.position.status == "long":
                self._close_position(current_data, "Reverse Close")

            margin_cost = price * self.lot_size * MARGIN_FRACTION
            if self.position.status == "flat" and self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("short", current_data)

        # action == 0 => no-op

    def _open_position(self, direction: str, current_data: pd.Series):
        entry_price = current_data.get('open', 0.0)
        margin_cost = entry_price * self.lot_size * MARGIN_FRACTION
        self.current_capital -= (margin_cost + self.transaction_cost)
        self.position.margin_used = margin_cost

        sl_val = current_data.get('StopLoss', 0.0)
        tg_val = current_data.get('Target', 0.0)
        direction_int = 1 if direction == "long" else -1

        self.position.open(
            entry_price=entry_price,
            sl_points=sl_val,
            target_points=tg_val,
            direction=direction_int,
            quantity=self.lot_size
        )

        self.trade_log.append({
            "entry_time": current_data.name if hasattr(current_data, 'name') else None,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None,
            "initial_capital": self._format_currency(self.initial_capital),
            "points": None,
            "profit": None,
            "win": None,
            "reward": None,
            "time_in_trade": None
        })

    # ==============
    # REWARDS
    # ==============
    def _apply_guidance_reward(self, action, guidance_row):
        if guidance_row.empty:
            return 0.0
        signal = guidance_row.get('Signal', 0)
        candles_to_profit = guidance_row.get('candles_to_profit', 1.0)
        candles_to_loss = guidance_row.get('candles_to_loss', 1.0)
        if candles_to_profit <= 0:
            candles_to_profit = 1.0
        if candles_to_loss <= 0:
            candles_to_loss = 1.0

        if signal == 1:
            correct_action = 1  # long
            guidance_reward = self.target_bonus / candles_to_profit
        elif signal == 2:
            correct_action = 3  # manual close
            guidance_reward = -self.sl_penalty / candles_to_loss
        elif signal == 3:
            correct_action = 2  # short
            guidance_reward = self.target_bonus / candles_to_profit
        elif signal == 4:
            correct_action = 3
            guidance_reward = -self.sl_penalty / candles_to_loss
        else:
            return 0.0

        if action == correct_action:
            return guidance_reward
        else:
            return -self.misalignment_penalty

    def _calculate_reward(self, unrealized_pnl: float, current_data: pd.Series) -> float:
        # 1) Risk-Adjusted PnL
        if self.position.sl_points == 0:
            risk_adj_pnl = 0.0
        else:
            risk = self.position.sl_points * self.lot_size
            risk_adj_pnl = (unrealized_pnl / risk) * self.unrealized_pnl_weight

        # 2) Time Penalty
        time_penalty = self.time_penalty_weight * (self.position.time_in_trade / 100.0)

        # 3) Volatility Penalty
        hi = current_data.get('high', 0.0)
        lo = current_data.get('low', 0.0)
        op = current_data.get('open', 1.0)
        if abs(op) < 1e-6:
            op = 1.0
        volatility = (hi - lo) / op
        volatility_penalty = self.volatility_penalty_weight * volatility

        # 4) SL Proximity
        if self.position.trailing_sl_price is not None:
            effective_sl = self.position.trailing_sl_price
        else:
            if self.position.direction == 1:
                effective_sl = self.position.entry_price - self.position.sl_points
            else:
                effective_sl = self.position.entry_price + self.position.sl_points

        if self.position.direction == 1:
            sl_range = abs(effective_sl - self.position.entry_price)
            sl_distance = max(0, current_data.get('close', 0.0) - effective_sl)
        else:
            sl_range = abs(effective_sl - self.position.entry_price)
            sl_distance = max(0, effective_sl - current_data.get('close', 0.0))

        sl_prox_ratio = min(sl_distance / sl_range, 1.0) if sl_range > 1e-6 else 1.0
        sl_proximity_penalty = self.sl_proximity_weight * (1.0 - sl_prox_ratio)

        # 5) Target Proximity
        target_proximity_bonus = 0.0
        if self.position.target_points > 0:
            direction = self.position.direction
            entry_price = self.position.entry_price
            if direction == 1:
                target_price = entry_price + self.position.target_points
                target_range = abs(target_price - entry_price)
                distance_to_target = max(0, target_price - current_data.get('close', 0.0))
                # Missed opportunity penalty if price has exceeded target
                if current_data.get('high', 0.0) >= target_price:
                    target_proximity_bonus -= self.missed_opportunity_penalty
            else:
                target_price = entry_price - self.position.target_points
                target_range = abs(entry_price - target_price)
                distance_to_target = max(0, current_data.get('close', 0.0) - target_price)
                if current_data.get('low', 0.0) <= target_price:
                    target_proximity_bonus -= self.missed_opportunity_penalty

            if target_range > 1e-6:
                ratio = 1.0 - (distance_to_target / target_range)
                target_proximity_bonus += ratio * self.target_proximity_weight

        # Combine
        total_reward = (
            risk_adj_pnl
            + target_proximity_bonus
            - time_penalty
            - volatility_penalty
            - sl_proximity_penalty
        )
        return float(np.clip(total_reward, -self.max_step_reward, self.max_step_reward))

    # ==============
    # POSITION CLOSE
    # ==============
    def _close_position(self, current_data: pd.Series, reason: str, override_exit_price=None) -> float:
        if self.position.status == "flat":
            return 0.0

        direction = self.position.direction
        if override_exit_price is not None:
            exit_price = override_exit_price
        elif "SL" in reason:
            if self.position.trailing_sl_price is not None:
                exit_price = self.position.trailing_sl_price
            else:
                if direction == 1:
                    exit_price = self.position.entry_price - self.position.sl_points
                else:
                    exit_price = self.position.entry_price + self.position.sl_points
        else:
            exit_price = current_data.get('open', 0.0)

        realized_pnl = (exit_price - self.position.entry_price) * direction * self.lot_size
        margin_return = self.position.margin_used
        self.current_capital += (margin_return + realized_pnl - self.transaction_cost)

        trade_return = realized_pnl / self.initial_capital
        total_reward = trade_return

        points = (exit_price - self.position.entry_price) * direction
        formatted_profit = self._format_currency(realized_pnl)
        win = realized_pnl > 0

        # Update the last opened trade
        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name if hasattr(current_data, 'name') else None,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason,
                    "time_in_trade": self.position.time_in_trade,
                    "points": round(points, 2),
                    "profit": formatted_profit,
                    "win": win,
                    "reward": total_reward
                })
                break

        self.position.reset()
        return total_reward

    # ==============
    # SL HIT CHECK
    # ==============
    def _check_sl_hit(self, current_data: pd.Series):
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        entry_price = self.position.entry_price
        current_open = current_data.get('open', 0.0)
        current_high = current_data.get('high', 0.0)
        current_low = current_data.get('low', 0.0)

        # Figure out effective SL
        if self.position.trailing_sl_price is not None:
            effective_sl = self.position.trailing_sl_price
        else:
            if direction == 1:
                effective_sl = entry_price - self.position.sl_points
            else:
                effective_sl = entry_price + self.position.sl_points

        if direction == 1:
            # If the bar opens below SL, or the intrabar low crosses SL
            if current_open <= effective_sl:
                return True, current_open
            elif current_low <= effective_sl:
                return True, effective_sl
        else:
            # If bar opens above SL, or intrabar high crosses SL
            if current_open >= effective_sl:
                return True, current_open
            elif current_high >= effective_sl:
                return True, effective_sl

        return False, None

    # ==============
    # TRAILING SL
    # ==============
    def _update_trailing_sl(self, current_data: pd.Series):
        direction = self.position.direction
        if direction == 1:
            if self.position.highest_price is None:
                self.position.highest_price = current_data.get('high', 0.0)
            else:
                self.position.highest_price = max(self.position.highest_price, current_data.get('high', 0.0))
            new_sl_price = self._calculate_trailing_sl_price(
                reference_price=self.position.highest_price,
                side="long"
            )
            init_sl = self.position.entry_price - self.position.sl_points
            if self.position.trailing_sl_price is not None:
                self.position.trailing_sl_price = max(self.position.trailing_sl_price, new_sl_price, init_sl)
            else:
                self.position.trailing_sl_price = max(new_sl_price, init_sl)
        else:
            if self.position.lowest_price is None:
                self.position.lowest_price = current_data.get('low', 1e9)
            else:
                self.position.lowest_price = min(self.position.lowest_price, current_data.get('low', 1e9))
            new_sl_price = self._calculate_trailing_sl_price(
                reference_price=self.position.lowest_price,
                side="short"
            )
            init_sl = self.position.entry_price + self.position.sl_points
            if self.position.trailing_sl_price is not None:
                self.position.trailing_sl_price = min(self.position.trailing_sl_price, new_sl_price, init_sl)
            else:
                self.position.trailing_sl_price = min(new_sl_price, init_sl)

    def _calculate_trailing_sl_price(self, reference_price: float, side: str) -> float:
        if self.trailing_sl_mode == "fixed":
            if side == "long":
                return reference_price - self.trailing_sl_amount
            else:
                return reference_price + self.trailing_sl_amount
        elif self.trailing_sl_mode == "percentage":
            offset = reference_price * self.trailing_sl_percent
            if side == "long":
                return reference_price - offset
            else:
                return reference_price + offset
        elif self.trailing_sl_mode == "factor":
            if side == "long":
                gain = reference_price - self.position.entry_price
                return self.position.entry_price + (gain * self.trailing_factor)
            else:
                gain = self.position.entry_price - reference_price
                return self.position.entry_price - (gain * self.trailing_factor)
        else:
            return reference_price

    # ==============
    # MARGIN CALL
    # ==============
    def _check_margin_call(self, current_price: float) -> bool:
        if self.position.status == "flat":
            return False

        maintenance_margin = self.position.margin_used * MARGIN_CALL_THRESHOLD
        unrealized_pnl = self.position.update_unrealized_pnl(current_price)
        equity = self.current_capital + unrealized_pnl

        return equity < maintenance_margin

    # ==============
    # METRICS
    # ==============
    def get_metrics(self) -> dict:
        if not self.trade_log:
            return {"instrument": self.instrument_name, "message": "No trades taken"}

        closed_trades = [t for t in self.trade_log if t['exit_time'] is not None]
        total_trades = len(closed_trades)
        if total_trades == 0:
            return {"instrument": self.instrument_name, "message": "No trades closed yet"}

        winning_trades = [t for t in closed_trades if t['pnl'] is not None and t['pnl'] > 0]
        losing_trades = [t for t in closed_trades if t['pnl'] is not None and t['pnl'] <= 0]

        win_rate = (len(winning_trades) / total_trades) * 100.0
        sum_winning = sum(t['pnl'] for t in winning_trades)
        sum_losing = sum(t['pnl'] for t in losing_trades)
        abs_sum_losing = abs(sum_losing) if sum_losing else 1e-9
        profit_factor = sum_winning / abs_sum_losing if abs_sum_losing > 1e-9 else float('inf')

        return {
            "instrument": self.instrument_name,
            "initial_capital": self._format_currency(self.initial_capital),
            "final_capital": self._format_currency(self.current_capital),
            "total_trades": total_trades,
            "win_rate": f"{round(win_rate, 2)}%",
            "total_reward": round(self.total_reward, 2),
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": profit_factor
        }

    def _compute_max_drawdown(self) -> float:
        equity_curve = [self.initial_capital]
        for trade in self.trade_log:
            if trade['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + trade['pnl'])
        equity_curve = np.array(equity_curve, dtype=np.float32)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return float(dd.max())

    def _format_currency(self, value: float) -> str:
        abs_value = abs(value)
        if abs_value >= 1e7:
            return f"₹{value / 1e7:.2f}Cr"
        elif abs_value >= 1e5:
            return f"₹{value / 1e5:.2f}L"
        elif abs_value >= 1e3:
            return f"₹{value / 1e3:.2f}K"
        else:
            return f"₹{value:.2f}"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import torch.multiprocessing as mp
import torch.utils.data as data
import torch.amp as amp
import torch.utils.checkpoint as cp
import torch.distributed as dist
from torch.nn.parallel import DataParallel
from torch.optim.lr_scheduler import CosineAnnealingLR


import numpy as np
import random
import time
import pickle
import os
import math

# fix the checkpoint warning by explicitly specifying use_reentrant=False
def checkpoint_forward(module, x):
    return module(x)

model_base_config = {
    # Model architecture
    "d_model": 1024,             # Larger hidden dim for more capacity
    "nhead": 16,                 # Adjust to 16 or 32; balanced for d_model=1024
    "num_layers": 12,            # Depth of the transformer
    "dim_feedforward": 8192,     # Bigger feed-forward layer for capacity
    "dropout": 0.1,
    "instrument_embed_dim": 64,

    "num_env_steps": 10_000_000,
    "rollout_steps": 2048,
    "grpo_epochs": 10,
    "mini_batch_size": 2048,
    "clip_param": 0.2,
    "lr": 1e-4,
    "entropy_coef": 0.01,
    "max_grad_norm": 0.5,
    "weight_decay": 1e-5,

    "stochastic_depth_prob": 0.2,

    "use_gpu": True,
    "log_interval": 1,
    "eval_interval": 50,

    # we will store experience in chunked files, not a single file
    # e.g. experience_memory_0.pkl, experience_memory_1.pkl, etc.
    # The class below will handle them automatically
    "memory_dir_path": "experience_chunks",
    "chunk_size": 50_000,       # number of experiences per chunk file
    "max_memory_size": 1_000_000, # total experiences to keep
    "early_stopping_patience": 5,
}

device = torch.device("cuda" if (torch.cuda.is_available() and model_base_config["use_gpu"]) else "cpu")
print(device)

cuda


In [ ]:
# %% [code]
class StochasticDepth(nn.Module):
    def __init__(self, p: float = 0.2):
        super(StochasticDepth, self).__init__()
        self.p = p

    def forward(self, x, residual):
        if self.training and torch.rand(1).item() < self.p:
            return x
        else:
            return x + residual

# %% [code]
class FFNFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout):
        super(FFNFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# %% [code]
class MoEFeedForward(nn.Module):
    def __init__(self, d_model, dim_feedforward, dropout, num_experts=4):
        super(MoEFeedForward, self).__init__()
        self.num_experts = num_experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, dim_feedforward),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(dim_feedforward, d_model)
            ) for _ in range(num_experts)
        ])
        self.gate = nn.Linear(d_model, num_experts)
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        gate_scores = self.softmax(self.gate(x))
        out_experts = []
        for i in range(self.num_experts):
            out_i = self.experts[i](x)
            out_experts.append(out_i.unsqueeze(-1))
        out_experts = torch.cat(out_experts, dim=-1)
        gate_scores = gate_scores.unsqueeze(2)
        moe_output = (out_experts * gate_scores).sum(dim=-1)
        return moe_output

# %% [code]
class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1, num_latent_tokens=4):
        super(MultiHeadLatentAttention, self).__init__()
        self.mha = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.latent_tokens = nn.Parameter(torch.randn(num_latent_tokens, d_model))
        self.layernorm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        latent_tokens_expanded = self.latent_tokens.unsqueeze(0).expand(B, -1, -1)
        x_cat = torch.cat([latent_tokens_expanded, x], dim=1)
        attn_output, _ = self.mha(x_cat, x_cat, x_cat)
        x_out = self.layernorm(x_cat + self.dropout(attn_output))
        return x_out[:, self.latent_tokens.size(0):, :]

# %% [code]
class DeepTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout, stochastic_depth_prob, use_moe=False):
        super(DeepTransformerLayer, self).__init__()
        self.attn = MultiHeadLatentAttention(d_model, nhead, dropout)
        self.layernorm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        if use_moe:
            self.ffn = MoEFeedForward(d_model, dim_feedforward, dropout)
        else:
            self.ffn = FFNFeedForward(d_model, dim_feedforward, dropout)

        self.layernorm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.stochastic_depth = StochasticDepth(stochastic_depth_prob)

    def forward(self, x):
        attn_out = self.attn(x)
        x = self.stochastic_depth(
            x=self.layernorm1(x + self.dropout1(attn_out)),
            residual=x
        )
        ffn_out = self.ffn(x)
        x = self.stochastic_depth(
            x=self.layernorm2(x + self.dropout2(ffn_out)),
            residual=x
        )
        return x

# %% [code]
class DeepTransformerCheckpointed(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout,
                 total_layers=24, stochastic_depth_prob=0.2, use_checkpointing=True):
        super(DeepTransformerCheckpointed, self).__init__()
        self.layers = nn.ModuleList()
        self.use_checkpointing = use_checkpointing

        # For example: 4 "regular" FFN layers, rest are MoE
        ffn_layers = 4
        moe_layers = total_layers - ffn_layers

        for _ in range(ffn_layers):
            self.layers.append(
                DeepTransformerLayer(
                    d_model, nhead, dim_feedforward, dropout,
                    stochastic_depth_prob=stochastic_depth_prob,
                    use_moe=False
                )
            )
        for _ in range(moe_layers):
            self.layers.append(
                DeepTransformerLayer(
                    d_model, nhead, dim_feedforward, dropout,
                    stochastic_depth_prob=stochastic_depth_prob,
                    use_moe=True
                )
            )

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpointing and self.training:
                if not x.requires_grad:
                    x = x.clone().detach().requires_grad_(True)
                x = cp.checkpoint(checkpoint_forward, layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x

# %% [code]
class MultiEncoder(nn.Module):
    def __init__(self, obs_dim, model_config, num_instruments=1):
        super(MultiEncoder, self).__init__()
        self.instrument_embed_dim = model_config["instrument_embed_dim"]
        self.instrument_embedding = nn.Embedding(num_instruments, self.instrument_embed_dim)
        self.input_linear = nn.Linear(obs_dim + self.instrument_embed_dim, model_config["d_model"])
        self.transformer_block = DeepTransformerCheckpointed(
            d_model=model_config["d_model"],
            nhead=model_config["nhead"],
            dim_feedforward=model_config["dim_feedforward"],
            dropout=model_config["dropout"],
            total_layers=model_config["num_layers"],
            stochastic_depth_prob=model_config["stochastic_depth_prob"],
            use_checkpointing=True
        )
        self.policy_head = nn.Linear(model_config["d_model"], 4)
        self.obs_dim = obs_dim
        self.d_model = model_config["d_model"]

    def forward(self, obs_dict, instrument_id):
        # obs_dict["history"] shape: (B, T, obs_dim)
        history = obs_dict["history"]
        B, seq_len, feat_dim = history.shape

        inst_emb = self.instrument_embedding(instrument_id)
        # inst_emb shape: (B, instrument_embed_dim)
        inst_emb_expanded = inst_emb.unsqueeze(1).expand(B, seq_len, self.instrument_embed_dim)

        x = torch.cat([history, inst_emb_expanded], dim=-1)
        x = self.input_linear(x)
        x = self.transformer_block(x)

        final_repr = x[:, -1, :]
        policy_logits = self.policy_head(final_repr)
        return policy_logits

# %% [code]
class ExternalMemory:
    def __init__(self, memory_dir_path, chunk_size=5000, max_memory_size=50000):
        self.memory_dir_path = memory_dir_path
        self.chunk_size = chunk_size
        self.max_memory_size = max_memory_size
        self.num_experiences = 0
        self.chunk_files = []

        os.makedirs(self.memory_dir_path, exist_ok=True)
        self._discover_existing_chunks()

    def _discover_existing_chunks(self):
        files = sorted([
            f for f in os.listdir(self.memory_dir_path)
            if f.startswith("experience_chunk_") and f.endswith(".pkl")
        ])
        self.chunk_files = [os.path.join(self.memory_dir_path, f) for f in files]
        self.num_experiences = 0
        for cf in self.chunk_files:
            with open(cf, "rb") as f:
                chunk_data = pickle.load(f)
                self.num_experiences += len(chunk_data)

    def add_experience(self, obs, action, logprob, reward, done, instrument_id):
        if len(self.chunk_files) == 0:
            chunk_path = os.path.join(self.memory_dir_path, "experience_chunk_0.pkl")
            with open(chunk_path, "wb") as f:
                pickle.dump([], f)
            self.chunk_files.append(chunk_path)

        last_chunk_path = self.chunk_files[-1]
        with open(last_chunk_path, "rb") as f:
            chunk_data = pickle.load(f)

        chunk_data.append((obs, action, logprob, reward, done, instrument_id))
        self.num_experiences += 1

        # pop from the oldest chunk if we exceed max memory
        if self.num_experiences > self.max_memory_size:
            first_chunk_path = self.chunk_files[0]
            with open(first_chunk_path, "rb") as f:
                old_chunk_data = pickle.load(f)
            if len(old_chunk_data) > 0:
                old_chunk_data.pop(0)
                self.num_experiences -= 1
                with open(first_chunk_path, "wb") as f:
                    pickle.dump(old_chunk_data, f)
                if len(old_chunk_data) == 0:
                    os.remove(first_chunk_path)
                    self.chunk_files.pop(0)
            else:
                os.remove(first_chunk_path)
                self.chunk_files.pop(0)

        # rotate chunk if we exceed chunk size
        if len(chunk_data) >= self.chunk_size:
            with open(last_chunk_path, "wb") as f:
                pickle.dump(chunk_data, f)
            new_chunk_index = len(self.chunk_files)
            new_chunk_path = os.path.join(self.memory_dir_path, f"experience_chunk_{new_chunk_index}.pkl")
            with open(new_chunk_path, "wb") as f:
                pickle.dump([], f)
            self.chunk_files.append(new_chunk_path)
        else:
            with open(last_chunk_path, "wb") as f:
                pickle.dump(chunk_data, f)

    def clear_memory(self):
        for cf in self.chunk_files:
            if os.path.exists(cf):
                os.remove(cf)
        self.chunk_files = []
        self.num_experiences = 0

    def get_chunk_paths(self):
        return self.chunk_files

    def total_experiences(self):
        return self.num_experiences

    def load_chunk_data(self, chunk_path):
        with open(chunk_path, "rb") as f:
            return pickle.load(f)

    def get_experience_generator(self, shuffle=True, batch_size=64):
        all_files = self.chunk_files[:]
        if shuffle:
            random.shuffle(all_files)

        for cf in all_files:
            chunk_data = self.load_chunk_data(cf)
            if shuffle:
                random.shuffle(chunk_data)
            for start in range(0, len(chunk_data), batch_size):
                end = start + batch_size
                yield chunk_data[start:end]

# %% [code]
def pad_or_truncate_history(history_array, target_dim):
    # ensures we match the max_obs_dim shape
    seq_len, feat_dim = history_array.shape
    if feat_dim == target_dim:
        return history_array
    elif feat_dim > target_dim:
        return history_array[:, :target_dim]
    else:
        pad_width = target_dim - feat_dim
        pad = np.zeros((seq_len, pad_width), dtype=np.float32)
        return np.concatenate([history_array, pad], axis=-1)

# %% [code]
class GRPORolloutBuffer:
    def __init__(self, rollout_size, device):
        self.rollout_size = rollout_size
        self.device = device
        self.reset()

    def reset(self):
        self.observations = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.dones = []
        self.instrument_ids = []
        self.advantages = []

    def add(self, obs, action, logprob, reward, done, instrument_id):
        self.observations.append(obs)
        self.actions.append(action)
        self.logprobs.append(logprob)
        self.rewards.append(reward)
        self.dones.append(done)
        self.instrument_ids.append(instrument_id)

    def compute_group_advantages(self):
        rewards_np = np.array(self.rewards, dtype=np.float32)
        mean_r = np.mean(rewards_np)
        adv = rewards_np - mean_r
        self.advantages = adv.tolist()

    def get_batches(self, mini_batch_size):
        indices = np.arange(len(self.observations))
        np.random.shuffle(indices)
        for start in range(0, len(self.observations), mini_batch_size):
            end = start + mini_batch_size
            batch_idx = indices[start:end]
            yield batch_idx

# %% [code]
class GRPOTrainer:
    def __init__(self, model_config):
        self.model_config = model_config
        self.clip_param = model_config["clip_param"]
        self.grpo_epochs = model_config["grpo_epochs"]
        self.mini_batch_size = model_config["mini_batch_size"]
        self.entropy_coef = model_config["entropy_coef"]
        self.max_grad_norm = model_config["max_grad_norm"]
        self.rollout_steps = model_config["rollout_steps"]

        self.policy_model = None
        self.optimizer = None
        self.scheduler = None
        self.external_memory = ExternalMemory(
            memory_dir_path=model_config["memory_dir_path"],
            chunk_size=model_config["chunk_size"],
            max_memory_size=model_config["max_memory_size"]
        )
        self.early_stopping_patience = model_config["early_stopping_patience"]
        self.epochs_no_improve = 0

        self.train_logs = {
            "episode_rewards": [],
            "grpo_losses": [],
            "best_val_reward": -float("inf")
        }

        if device.type == "cuda":
            self.scaler = amp.GradScaler("cuda")
        else:
            self.scaler = amp.GradScaler(enabled=False)

        self.max_obs_dim = None
        self.num_instruments = None

    def build_policy_model(self, obs_dim, num_instruments):
        self.max_obs_dim = obs_dim
        self.num_instruments = num_instruments
        base_model = MultiEncoder(
            obs_dim=obs_dim,
            model_config=self.model_config,
            num_instruments=num_instruments
        ).to(device)

        if torch.cuda.device_count() > 1:
            print("Using", torch.cuda.device_count(), "GPUs for DataParallel.")
            self.policy_model = DataParallel(base_model)
        else:
            self.policy_model = base_model

        self.optimizer = optim.Adam(
            self.policy_model.parameters(),
            lr=self.model_config["lr"],
            weight_decay=self.model_config["weight_decay"]
        )
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=100, eta_min=1e-6)

    def _fix_obs_dim(self, obs):
        old_hist = obs["history"]
        new_hist = pad_or_truncate_history(old_hist, self.max_obs_dim)
        fixed_obs = {"history": new_hist}
        return fixed_obs

    def _convert_obs(self, obs, instrument_id):
        obs = self._fix_obs_dim(obs)
        obs_dict_t = {
            "history": torch.tensor(obs["history"], dtype=torch.float32, device=device).unsqueeze(0)
        }
        inst_id_t = torch.tensor([instrument_id], dtype=torch.long, device=device)
        return obs_dict_t, inst_id_t

    def _collect_rollout(self, env):
        # We now fetch the global instrument_id from the environment itself
        instrument_id = env.instrument_id
        buffer = GRPORolloutBuffer(self.rollout_steps, device)
        obs, _ = env.reset()
        obs = self._fix_obs_dim(obs)
        done = False
        total_reward = 0.0

        for step in range(self.rollout_steps):
            obs_dict_t, inst_id_t = self._convert_obs(obs, instrument_id)
            with torch.no_grad():
                logits = self.policy_model(obs_dict_t, inst_id_t)
                dist = Categorical(logits=logits)
                action = dist.sample()
                logprob = dist.log_prob(action).item()

            action_int = action.item()
            next_obs, reward, done, _, info = env.step(action_int)
            next_obs = self._fix_obs_dim(next_obs)
            total_reward += reward

            buffer.add(obs, action_int, logprob, reward, done, instrument_id)
            self.external_memory.add_experience(obs, action_int, logprob, reward, done, instrument_id)

            obs = next_obs
            if done:
                obs, _ = env.reset()
                obs = self._fix_obs_dim(obs)

        buffer.compute_group_advantages()
        return buffer, total_reward

    def _update_grpo(self, buffer):
        obs_arr = buffer.observations
        actions_arr = buffer.actions
        old_logprobs_arr = buffer.logprobs
        advantages_arr = buffer.advantages
        instrument_ids_arr = buffer.instrument_ids

        old_logprobs_tensor = torch.tensor(old_logprobs_arr, dtype=torch.float32, device=device)
        advantages_tensor = torch.tensor(advantages_arr, dtype=torch.float32, device=device)
        actions_tensor = torch.tensor(actions_arr, dtype=torch.long, device=device)
        advantages_tensor = (advantages_tensor - advantages_tensor.mean()) / (advantages_tensor.std() + 1e-8)

        total_policy_loss = 0.0
        total_entropy = 0.0

        for epoch_i in range(self.grpo_epochs):
            for batch_idx in buffer.get_batches(self.mini_batch_size):
                obs_batch = [obs_arr[i] for i in batch_idx]
                actions_batch = actions_tensor[batch_idx]
                old_logprobs_batch = old_logprobs_tensor[batch_idx]
                advantages_batch = advantages_tensor[batch_idx]
                inst_id_batch_np = [instrument_ids_arr[i] for i in batch_idx]

                obs_dict_list = []
                inst_id_list = []
                for ob, inst_id in zip(obs_batch, inst_id_batch_np):
                    ob_dict_t, inst_id_t = self._convert_obs(ob, inst_id)
                    obs_dict_list.append(ob_dict_t)
                    inst_id_list.append(inst_id_t)

                history_batch = torch.cat([o["history"] for o in obs_dict_list], dim=0)
                inst_id_batch_tensor = torch.cat(inst_id_list, dim=0)
                batch_obs_dict = {"history": history_batch}

                with amp.autocast(device_type=device.type, enabled=(device.type=="cuda")):
                    logits = self.policy_model(batch_obs_dict, inst_id_batch_tensor)
                    dist = Categorical(logits=logits)
                    logprobs = dist.log_prob(actions_batch)
                    entropy = dist.entropy().mean()

                    ratio = torch.exp(logprobs - old_logprobs_batch)
                    surr1 = ratio * advantages_batch
                    surr2 = torch.clamp(ratio, 1.0 - self.clip_param, 1.0 + self.clip_param) * advantages_batch
                    policy_loss = -torch.min(surr1, surr2).mean()
                    loss = policy_loss - self.entropy_coef * entropy

                self.optimizer.zero_grad()
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.policy_model.parameters(), self.max_grad_norm)
                self.scaler.step(self.optimizer)
                self.scaler.update()

                total_policy_loss += policy_loss.item()
                total_entropy += entropy.item()

            # step the scheduler after each epoch
            self.scheduler.step()

        n_updates = self.grpo_epochs * max(1, (len(obs_arr) // self.mini_batch_size))
        avg_policy_loss = total_policy_loss / max(1, n_updates)
        avg_entropy = total_entropy / max(1, n_updates)
        return avg_policy_loss, avg_entropy

    def evaluate(self, env, n_eval_episodes=3):
        # load best checkpoint if you wish to test purely from best
        # but you can also skip this if you want a continuous measure
        # if os.path.exists("best_grpo_model.pt"):
        #     self.load_checkpoint("best_grpo_model.pt")

        instrument_id = env.instrument_id
        total_eval_reward = 0.0
        for _ in range(n_eval_episodes):
            obs, _ = env.reset()
            obs = self._fix_obs_dim(obs)
            done = False
            episode_reward = 0.0
            while not done:
                obs_dict_t, inst_id_t = self._convert_obs(obs, instrument_id)
                with torch.no_grad():
                    logits = self.policy_model(obs_dict_t, inst_id_t)
                    dist = Categorical(logits=logits)
                    action = dist.sample().item()
                obs, reward, done, _, info = env.step(action)
                obs = self._fix_obs_dim(obs)
                episode_reward += reward
            total_eval_reward += episode_reward
        return total_eval_reward / n_eval_episodes

    def save_checkpoint(self, path):
        model_to_save = self.policy_model.module if hasattr(self.policy_model, "module") else self.policy_model
        checkpoint = {
            "model_state_dict": model_to_save.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "train_logs": self.train_logs,
            "scheduler_state_dict": self.scheduler.state_dict()
        }
        torch.save(checkpoint, path)

    def load_checkpoint(self, path):
        checkpoint = torch.load(path, map_location=device)
        model_to_load = self.policy_model.module if hasattr(self.policy_model, "module") else self.policy_model
        model_to_load.load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if "scheduler_state_dict" in checkpoint:
            self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        self.train_logs = checkpoint["train_logs"]

    def train_one_chunk(self, envs, steps_for_this_chunk):
        steps_collected = 0
        rollout_count = 0

        while steps_collected < steps_for_this_chunk:
            rollout_count += 1

            # random selection among the chunk's envs
            env = random.choice(envs)
            buffer, rollout_reward = self._collect_rollout(env)
            steps_collected += self.rollout_steps

            policy_loss, entropy = self._update_grpo(buffer)
            self.train_logs["grpo_losses"].append((policy_loss, entropy))
            self.train_logs["episode_rewards"].append(rollout_reward)

            metrics = env.get_metrics() if hasattr(env, "get_metrics") else {}
            if rollout_count % self.model_config["log_interval"] == 0:
                print(f"Rollout: {rollout_count}, Steps in chunk: {steps_collected}, "
                      f"Instrument: {metrics.get('instrument','Unknown')} (ID={env.instrument_id}), Reward: {rollout_reward:.2f}, "
                      f"InitCap: {metrics.get('initial_capital','N/A')}, "
                      f"FinalCap: {metrics.get('final_capital','N/A')}, "
                      f"Trades: {metrics.get('total_trades',0)}, WinRate: {metrics.get('win_rate','N/A')}, "
                      f"PolicyLoss: {policy_loss:.4f}, Entropy: {entropy:.4f}")

            # occasional validation
            if (self.model_config["eval_interval"] > 0) and (rollout_count % self.model_config["eval_interval"] == 0):
                val_env = random.choice(envs)
                val_reward = self.evaluate(val_env, n_eval_episodes=3)
                val_metrics = val_env.get_metrics() if hasattr(val_env, "get_metrics") else {}
                print(f"[Validation] Instrument: {val_metrics.get('instrument','Unknown')} (ID={val_env.instrument_id}), "
                      f"AvgReward: {val_reward:.2f}, InitCap: {val_metrics.get('initial_capital','N/A')}, "
                      f"FinalCap: {val_metrics.get('final_capital','N/A')}, "
                      f"Trades: {val_metrics.get('total_trades',0)}, WinRate: {val_metrics.get('win_rate','N/A')}")

                if val_reward > self.train_logs["best_val_reward"]:
                    self.train_logs["best_val_reward"] = val_reward
                    self.epochs_no_improve = 0
                    self.save_checkpoint("best_grpo_model.pt")
                    print(f"New best reward: {val_reward:.2f}. Checkpoint saved.")
                else:
                    self.epochs_no_improve += 1
                    if self.epochs_no_improve >= self.early_stopping_patience:
                        print(f"Early stopping triggered! No improvement for {self.early_stopping_patience} validations.")
                        return steps_collected
        return steps_collected

    def train_in_chunks(self, global_config, chunk_size=10, total_env_steps=None, create_env_fn=None):
        if total_env_steps is None:
            total_env_steps = self.model_config["num_env_steps"]

        full_instrument_list = global_config["instruments"]

        # Ensure each instrument dict has a valid "instrument_id"
        # We'll just assign a global index if not present
        for idx, inst_conf in enumerate(full_instrument_list):
            if "instrument_id" not in inst_conf:
                inst_conf["instrument_id"] = idx

        def chunkify(lst, chunk_sz):
            for i in range(0, len(lst), chunk_sz):
                yield lst[i:i + chunk_sz]

        instrument_chunks = list(chunkify(full_instrument_list, chunk_size))

        if create_env_fn is None:
            raise ValueError("Please provide 'create_env_fn' to build your environment.")
        if len(instrument_chunks) == 0 or len(instrument_chunks[0]) == 0:
            raise ValueError("No instrument configs found in global_config['instruments']")

        # find max_obs_dim across all instruments
        max_obs_dim = 0
        envs_temp = []
        for inst_conf in full_instrument_list:
            merged_conf_sample = global_config.copy()
            merged_conf_sample.update(inst_conf)
            tmp_env = create_env_fn(merged_conf_sample)
            obs, _ = tmp_env.reset()
            obs_dim_local = obs["history"].shape[-1]
            if obs_dim_local > max_obs_dim:
                max_obs_dim = obs_dim_local
            envs_temp.append(tmp_env)

        self.build_policy_model(max_obs_dim, len(full_instrument_list))
        del envs_temp

        # optionally load existing best checkpoint
        if os.path.exists("best_grpo_model.pt"):
            self.load_checkpoint("best_grpo_model.pt")
            print("Checkpoint loaded. Resuming training from best_grpo_model.pt")

        steps_done = 0
        chunk_index = 0

        while steps_done < total_env_steps:
            for chunk_instruments in instrument_chunks:
                envs_for_chunk = []
                for inst_conf in chunk_instruments:
                    env_config = global_config.copy()
                    env_config.update(inst_conf)
                    envs_for_chunk.append(create_env_fn(env_config))

                steps_for_this_chunk = min(
                    self.rollout_steps * len(chunk_instruments) * 3,
                    total_env_steps - steps_done
                )

                chunk_steps_collected = self.train_one_chunk(envs_for_chunk, steps_for_this_chunk)
                steps_done += chunk_steps_collected
                print(f"Chunk {chunk_index} done: collected {chunk_steps_collected} steps, total so far {steps_done}")

                # cleanup envs
                for e in envs_for_chunk:
                    e.close()
                del envs_for_chunk

                if steps_done >= total_env_steps:
                    break
                chunk_index += 1

            if steps_done < total_env_steps:
                print(f"Completed full pass of chunks. Steps done: {steps_done}/{total_env_steps}.")
                chunk_index = 0

        print(f"Training finished. Total steps done: {steps_done}")

In [ ]:
def create_env_fn(env_config):
    return TradingEnv(env_config)

trainer = GRPOTrainer(model_base_config)

# Start training in chunks
trainer.train_in_chunks(
    global_config=config,
    chunk_size=10,
    total_env_steps=model_base_config["num_env_steps"],
    create_env_fn=create_env_fn
)